# Views: core gold (gold_event, gold_person_life)

Source-controlled definitions of the two views the whole gold layer reads. Split out of
`07_materialise_gold_tables` so the data refresh there never recreates views.

## Why views live in their own notebooks
`CREATE OR REPLACE VIEW` drops and recreates the view, which makes the **executing identity its
owner**. A view owned by someone else that reads it (e.g. `gold_person_evidence_summary`) then
fails with `INSUFFICIENT_PERMISSIONS`. So view DDL is kept out of the data-refresh notebooks:
views are *definitions*, not data, and only need (re)creating when a definition changes.

**Run this notebook only when a view definition changes**, not as part of a routine refresh.
Run it as the account that owns the views (or follow it with
`ALTER VIEW <name> OWNER TO <owner>` for each view it recreates).

## Run order
After Notebook 05a (`silver_event_date`) and Notebook 04. `gold_event` first, then
`gold_person_life` (reads it). After a change, re-run `07_materialise_gold_tables` so
`gold_person_event_timeline` and everything downstream is rebuilt.

## History
- 2026-09-15: BEF fix (date_end for BEF), Asana 1218448497347847.
- 2026-10-11: ABT fix (midpoint of the +/-1yr range, not date_start), Asana 1218492852523020.


In [0]:
%sql
-- Source-controls genealogy.gold_event (previously defined only in Databricks,
-- flagged in Asana 1218420159653198 as "cannot revert if view changed").
-- Fix (2026-09-15): event_date_parsed/event_year_parsed used to take
-- d.date_start unconditionally. That's correct for EXACT/ABT/AFT/RANGE dates
-- (date_start holds the real parsed value there) but wrong for BEF dates --
-- silver_event_date deliberately encodes "BEF <year>" as an open range
-- (date_start = year 1, the unbounded lower end; date_end = <year>, the known
-- upper bound), so taking date_start on a BEF row produced year 1 instead of
-- the real year. Traced from DQ-001/DQ-009 violations (death_year=1) via
-- Asana 1218448497347847 -- David Edmiston, Jonathan Croxon and others
-- workspace-wide. Now takes date_end for BEF rows, date_start for everything
-- else (unchanged behaviour for EXACT/AFT/RANGE, verified in Cell 3).
-- Fix (2026-10-11, DQ-009 / Asana 1218492852523020): ABT is NOT the real parsed
-- value at date_start either. 05a_parse_dates widens "abt YYYY" to a +/-1 year
-- range (date_start = Y-1-01-01, date_end = Y+1-12-31), so date_start made every
-- ABT year one year early (~2,000 events). ABT now resolves to the midpoint of
-- the symmetric range, i.e. the stated date (zero-day DAY-precision ranges
-- resolve to the date itself).
CREATE OR REPLACE VIEW genealogy.gold_event (
  record_xref,
  record_type,
  event_line_no,
  event_value,
  event_id,
  event_type,
  event_subtype,
  event_date_raw,
  event_date_parsed,
  event_year_parsed,
  event_date_type,
  event_place,
  event_country_inferred,
  event_ordinal_within_type,
  event_sequence_by_date,
  event_sequence_in_record)
WITH SCHEMA COMPENSATION
AS SELECT
  e.record_xref,
  CASE WHEN e.record_xref LIKE '%F%' THEN 'F' ELSE 'P' END AS record_type,
  e.event_line_no,
  e.event_value,
  e.event_id,
  e.event_type,
  e.event_subtype,
  e.event_date AS event_date_raw,
  -- BEF fix: date_end holds the meaningful bound for a BEF-type range.
  -- ABT fix: midpoint of the +/-1 year widening is the stated date.
  CASE WHEN d.date_type = 'BEF' THEN d.date_end
        WHEN d.date_type = 'ABT' THEN DATE_ADD(d.date_start, CAST(DATEDIFF(d.date_end, d.date_start) / 2 AS INT))
        ELSE d.date_start END AS event_date_parsed,
  year(CASE WHEN d.date_type = 'BEF' THEN d.date_end
        WHEN d.date_type = 'ABT' THEN DATE_ADD(d.date_start, CAST(DATEDIFF(d.date_end, d.date_start) / 2 AS INT))
        ELSE d.date_start END) AS event_year_parsed,
  d.date_type AS event_date_type,
  pc.preferred_name AS event_place,
  pc.inferred_country AS event_country_inferred,
  e.event_ordinal_within_type,
  -- NOTE: still ordered on d.date_start, unchanged from the pre-fix view --
  -- a BEF event's date_start is still the year-1 sentinel, so event
  -- sequencing for BEF-dated events is not corrected by this change. Out of
  -- scope for the DQ-001 date-plausibility bug this fix addresses; flag
  -- separately if BEF event ordering needs the same treatment.
  row_number() OVER (
      PARTITION BY e.record_xref
      ORDER BY d.date_start, e.event_type
    ) AS event_sequence_by_date,
  event_sequence_in_record
FROM
    genealogy.silver_event e
    LEFT JOIN genealogy.silver_event_date d
      ON e.event_id = d.event_id
    LEFT JOIN genealogy.silver_place_variant pv
      ON e.event_place_variant_id = pv.place_variant_id
    LEFT JOIN genealogy.silver_v_active_place_canonical pc
      ON pc.canonical_place_id = pv.canonical_place_id
WHERE event_type NOT IN ('NAME', 'SEX', 'FACT')


In [0]:
%sql
-- Source-controls genealogy.gold_person_life (previously defined only in
-- Databricks, flagged in Asana 1218420159653198). Logic unchanged from the
-- prior production definition -- captured as-is. Inherits the Cell 1 BEF fix
-- automatically since birth_date/death_date come straight from gold_event.
CREATE OR REPLACE VIEW genealogy.gold_person_life (
  person_gedcom_id,
  given_name,
  first_name,
  surname,
  display_name,
  sex,
  birth_date,
  birth_year,
  birth_date_raw,
  birth_date_type,
  birth_place,
  death_date,
  death_year,
  death_date_raw,
  death_place,
  age_at_death,
  has_birth_event,
  has_death_event,
  is_living)
WITH SCHEMA COMPENSATION
AS WITH ranked_events AS (
  SELECT
    ge.record_xref as person_id,
    ge.event_id,
    ge.event_type,
    ge.event_date_raw,
    ge.event_date_parsed,
    ge.event_date_type,
    ge.event_place as place
  FROM
    genealogy.gold_event ge
  WHERE
    event_type IN ('BIRT', 'DEAT')
    AND event_ordinal_within_type = 1
),
life_events AS (
  SELECT
    person_id,
    MAX(
      CASE
        WHEN event_type = 'BIRT' THEN event_id
      END
    ) AS birth_event_id,
    MAX(
      CASE
        WHEN event_type = 'BIRT' THEN event_date_parsed
      END
    ) AS birth_date,
    MAX(
      CASE
        WHEN event_type = 'BIRT' THEN event_date_raw
      END
    ) AS birth_date_raw,
    MAX(
      CASE
        WHEN event_type = 'BIRT' THEN event_date_type
      END
    ) AS birth_date_type,
    MAX(
      CASE
        WHEN event_type = 'BIRT' THEN place
      END
    ) AS birth_place,
    MAX(
      CASE
        WHEN event_type = 'DEAT' THEN event_id
      END
    ) AS death_event_id,
    MAX(
      CASE
        WHEN event_type = 'DEAT' THEN event_date_parsed
      END
    ) AS death_date,
    MAX(
      CASE
        WHEN event_type = 'DEAT' THEN event_date_raw
      END
    ) AS death_date_raw,
    MAX(
      CASE
        WHEN event_type = 'DEAT' THEN place
      END
    ) AS death_place
  FROM
    ranked_events
  GROUP BY
    person_id
)
SELECT
  p.person_gedcom_id,
  n.given_name,
  split_part(n.given_name, ' ', 1) as first_name,
  n.surname,
  TRIM(CONCAT(COALESCE(n.given_name, ''), ' ', COALESCE(n.surname, ''))) AS display_name,
  p.sex,
  le.birth_date,
  year(birth_date) as birth_year,
  le.birth_date_raw,
  le.birth_date_type,
  le.birth_place,
  le.death_date,
  year(death_date) as death_year,
  le.death_date_raw,
  le.death_place,
  CASE
    WHEN
      le.birth_date IS NOT NULL
      AND le.death_date IS NOT NULL
    THEN
      floor(months_between(le.death_date, le.birth_date) / 12)
  END AS age_at_death,
  le.birth_event_id IS NOT NULL AS has_birth_event,
  le.death_event_id IS NOT NULL AS has_death_event,
  case
    when
      year(current_date) - year(birth_date) <= 100
      AND le.death_date IS NULL
    then
      1
    else 0
  end as is_living
FROM
  genealogy.silver_person p
    LEFT JOIN genealogy.silver_person_name n
      ON p.person_gedcom_id = n.person_gedcom_id
    LEFT JOIN life_events le
      ON p.person_gedcom_id = le.person_id
